# 10 - Top Aday Modeller: Sabit 0.50 Esik ile Yeniden Dogrulama

**Goal:** Notebook 09'da bazi modellerin `test_mcc = 0` cikmasinin sebebi MCC-tabanli esik aramasinin **degenerate noktaya (her seyi Bull tahmin etme)** kaymasiydi. Bu notebook ayni modelleri **`threshold_search.enabled = False`** ve sabit 0.50 esikle yeniden calistirir. Boylece her aday modelin **gercek karar gucu** olcum edilir, esik gurultusunden arindirilmis halde.

**Aday secim kriterleri** (notebook 09 sonuclarindan):

| Aday | Secim sebebi |
|---|---|
| `cnn1d c128_k3_l2_d2`         | nb09'da en yuksek CV MCC (0.1346) - test_mcc=0 cikan ana suphe |
| `gru h64_l2_d2`                | nb09'da en tutarli (CV=0.041, Test=0.043) - kontrol |
| `cnn1d c64_k5_l2_d2`           | nb09'da en yuksek PR-AUC (0.6510) - olasilik siralamasi temiz |
| `transformer d64_h4_l1_d2`     | nb09'da degenerate olmayan en iyi test MCC (0.108) |

**Output isolation:** Tum cikti `threshold_free_results/` altina (baseline `artifacts/` ve `top_3_results/` ile karismaz). Colab'da `Drive/ANN-Project/Threshold_Free_Results/` altina mirror edilir.

---

## 0 - Imports & setup

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    TOOLKIT_PATH = '/content/drive/MyDrive/ANN-Project/toolkit.py'
    exec(open(TOOLKIT_PATH).read())
    setup()
    ROOT = Path('/content/repo')
else:
    ROOT = Path('..').resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

import json
import shutil
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.run_experiment import main as run_main

CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
RESULTS_DIR = ROOT / 'threshold_free_results'
RESULTS_DIR.mkdir(exist_ok=True)

NB09_RESULTS_CSV = ROOT / 'top_3_results' / 'sweep_results.csv'

print(f'ROOT             : {ROOT}')
print(f'RESULTS_DIR      : {RESULTS_DIR}')
print(f'NB09 sonuclari   : {NB09_RESULTS_CSV} ({"var" if NB09_RESULTS_CSV.exists() else "yok"})')

---
## 1 - Aday modeller

Her aday icin tam config (model, arch, scaler, quantile, lookback) hardcoded - nb09 ile birebir ayni hyperparametreler. Tek fark esik aramasi kapali olacak.

In [ ]:
CANDIDATES = [
    {
        'model'    : 'cnn1d',
        'arch'     : {'conv_channels': 128, 'kernel_size': 3, 'num_conv_layers': 2, 'dropout': 0.2},
        'scaler'   : 'standard',
        'quantile' : 0.40,
        'lookback' : 42,
        'label'    : 'c128_k3_l2_d2',
        'reason'   : 'nb09 en yuksek CV MCC (0.1346) - degenerate test_mcc',
    },
    {
        'model'    : 'gru',
        'arch'     : {'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.2},
        'scaler'   : 'robust',
        'quantile' : 0.20,
        'lookback' : 42,
        'label'    : 'h64_l2_d2',
        'reason'   : 'nb09 en tutarli (CV=0.041, Test=0.043) - control',
    },
    {
        'model'    : 'cnn1d',
        'arch'     : {'conv_channels': 64, 'kernel_size': 5, 'num_conv_layers': 2, 'dropout': 0.2},
        'scaler'   : 'standard',
        'quantile' : 0.40,
        'lookback' : 42,
        'label'    : 'c64_k5_l2_d2',
        'reason'   : 'nb09 en yuksek PR-AUC (0.6510) - olasilik temiz',
    },
    {
        'model'    : 'transformer_encoder',
        'arch'     : {'d_model': 64, 'nhead': 4, 'num_layers': 1, 'dim_feedforward': 128, 'dropout': 0.2},
        'scaler'   : 'robust',
        'quantile' : 0.20,
        'lookback' : 5,
        'label'    : 'd64_h4_l1_d2',
        'reason'   : 'nb09 degenerate olmayan en iyi test_mcc (0.108)',
    },
]

print(f'Toplam {len(CANDIDATES)} aday yeniden calistirilacak.\n')
for i, c in enumerate(CANDIDATES, 1):
    print(f"{i}. {c['model']}/{c['label']}")
    print(f"   scaler={c['scaler']}, q={c['quantile']}, lb={c['lookback']}")
    print(f"   arch  = {c['arch']}")
    print(f"   sebep = {c['reason']}\n")

---
## 2 - Sabit esikle yeniden calistirma

Her aday icin tek override farki: `decision.threshold_search.enabled = False`. Bu durumda model her zaman `probability_threshold = 0.50` ile karar verir. Eger nb09'da `test_mcc = 0` cikan adaylar burada anlamli bir test MCC verirse, sorun gercekten esik kalibrasyonuydu.

In [ ]:
results = []

for c in CANDIDATES:
    model = c['model']
    scaler = c['scaler']
    quantile = float(c['quantile'])
    lookback = int(c['lookback'])
    label = c['label']
    arch = c['arch']
    exp_name = f'tf_{model}_{label}'

    overrides = {
        'experiment': {'name': exp_name},
        'models': {'enabled': [model]},
        'preprocessing': {'scalers': [scaler]},
        'labeling': {
            'threshold_quantile': quantile,
            'threshold_quantile_candidates': [quantile],
        },
        'sequence': {
            'lookback': lookback,
            'lookback_candidates': [lookback],
        },
        'decision': {
            'probability_threshold': 0.50,
            'threshold_search': {'enabled': False},
        },
        'artifacts': {'root_dir': 'threshold_free_results/artifacts'},
        model: arch,
    }

    print(f'\n=== {exp_name} ===')
    print(f'    arch = {arch}')
    try:
        report = run_main(str(CONFIG_PATH), config_overrides=overrides)
        cv_summary = report['best_cv_selection']['cv_summary']
        test_metrics = report['final_test']['metrics']
        selected_thr = report['final_test'].get('selected_probability_threshold')
        row = {
            'model'         : model,
            'variant'       : label,
            'arch'          : str(arch),
            'fixed_threshold': selected_thr,
            'cv_mcc'        : cv_summary.get('mcc_mean'),
            'cv_mcc_std'    : cv_summary.get('mcc_std'),
            'cv_f1'         : cv_summary.get('f1_mean'),
            'cv_bal_acc'    : cv_summary.get('balanced_accuracy_mean'),
            'test_mcc'      : test_metrics.get('mcc'),
            'test_f1'       : test_metrics.get('f1'),
            'test_bal_acc'  : test_metrics.get('balanced_accuracy'),
            'test_pr_auc'   : test_metrics.get('pr_auc'),
            'test_roc_auc'  : test_metrics.get('roc_auc'),
        }
        results.append(row)
        print(f'    OK  CV_MCC={row["cv_mcc"]:.4f}  TEST_MCC={row["test_mcc"]:.4f}  esik={selected_thr}')
    except Exception as exc:
        print(f'    FAIL  {type(exc).__name__}: {exc}')
        traceback.print_exc()
        results.append({
            'model': model, 'variant': label, 'arch': str(arch),
            'error': f'{type(exc).__name__}: {exc}',
        })

results_df = pd.DataFrame(results)
results_csv = RESULTS_DIR / 'threshold_free_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'\nTamamlandi: {len(results)} run. Kaydedildi: {results_csv}')

---
## 3 - Notebook 09 ile karsilastirma

Her aday icin **(nb09 esik-aramali sonuc) vs (nb10 sabit-0.50)** yan yana. Hipotez:

- `test_mcc = 0` cikan adaylarda nb10 sabit esikle anlamli (>0) test MCC vermeli -> **esik gurultusu suclu**
- Yine `test_mcc = 0` cikiyorsa modelin temelde sinyali yok -> **veri/mimari suclu**

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.float_format', lambda v: f'{v:.4f}' if pd.notna(v) else 'N/A')

if NB09_RESULTS_CSV.exists():
    nb09 = pd.read_csv(NB09_RESULTS_CSV)
    nb09_subset = nb09[['model', 'variant', 'cv_mcc', 'test_mcc', 'test_f1', 'test_bal_acc', 'test_pr_auc']].copy()
    nb09_subset.columns = ['model', 'variant', 'nb09_cv_mcc', 'nb09_test_mcc', 'nb09_test_f1', 'nb09_test_bal_acc', 'nb09_test_pr_auc']

    nb10_subset = results_df[['model', 'variant', 'cv_mcc', 'test_mcc', 'test_f1', 'test_bal_acc', 'test_pr_auc']].copy()
    nb10_subset.columns = ['model', 'variant', 'nb10_cv_mcc', 'nb10_test_mcc', 'nb10_test_f1', 'nb10_test_bal_acc', 'nb10_test_pr_auc']

    cmp = nb09_subset.merge(nb10_subset, on=['model', 'variant'], how='inner')
    cmp['delta_test_mcc'] = cmp['nb10_test_mcc'] - cmp['nb09_test_mcc']
    cmp['delta_cv_mcc']   = cmp['nb10_cv_mcc']   - cmp['nb09_cv_mcc']

    cmp = cmp[[
        'model', 'variant',
        'nb09_cv_mcc', 'nb10_cv_mcc', 'delta_cv_mcc',
        'nb09_test_mcc', 'nb10_test_mcc', 'delta_test_mcc',
        'nb09_test_pr_auc', 'nb10_test_pr_auc',
    ]]

    print('=== nb09 (esik-arama) vs nb10 (sabit 0.50) ===\n')
    print(cmp.to_string(index=False))

    print('\n=== Yorum (delta_test_mcc):')
    for _, row in cmp.iterrows():
        d = row['delta_test_mcc']
        tag = 'IYI (esik gurultusu suclu)' if d > 0.02 else ('AYNI (degisim yok)' if abs(d) < 0.02 else 'KOTU (sabit esik daha kotu)')
        print(f"  {row['model']:<22} {row['variant']:<18} delta_test_mcc = {d:+.4f}  {tag}")
else:
    print(f'NB09 sonuc CSV yok ({NB09_RESULTS_CSV}). Sadece nb10 sonuclari gosteriliyor.')
    print(results_df.to_string(index=False))

---
## 4 - Siralama (nb10 sonuclari)

Sabit esikli yeniden run'da CV MCC ve Test MCC'ye gore siralama.

In [ ]:
if 'cv_mcc' not in results_df.columns:
    print('Sonuc yok / hata.')
else:
    ok = results_df.dropna(subset=['cv_mcc'])
    print('=== CV MCC azalan ===')
    cols = ['model', 'variant', 'cv_mcc', 'cv_mcc_std', 'test_mcc', 'test_f1', 'test_bal_acc', 'test_pr_auc']
    print(ok.sort_values('cv_mcc', ascending=False)[cols].to_string(index=False))

    print('\n=== Test MCC azalan ===')
    print(ok.sort_values('test_mcc', ascending=False)[cols].to_string(index=False))

    print('\n=== PR-AUC azalan (esikten bagimsiz, en temiz metrik) ===')
    print(ok.sort_values('test_pr_auc', ascending=False)[cols].to_string(index=False))

---
## 5 - Loss / val_loss egrileri

Her aday icin 4 fold loss egrisi. Notebook 09'daki ile ayni format.

In [ ]:
metrics_dir = RESULTS_DIR / 'artifacts' / 'metrics'

def load_fold_history(exp_name):
    matches = sorted(metrics_dir.glob(f'{exp_name}_*_folds.json'))
    if not matches:
        return None, None
    folds_path = matches[0]
    with open(folds_path, encoding='utf-8') as f:
        data = json.load(f)
    return data.get('fold_results', []), folds_path

def plot_loss_for_experiment(exp_name, show=True):
    folds, folds_path = load_fold_history(exp_name)
    if folds is None:
        print(f'  [SKIP] {exp_name}: folds JSON bulunamadi.')
        return
    n_folds = len(folds)
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 3.5), sharey=True)
    if n_folds == 1:
        axes = [axes]
    for i, fold in enumerate(folds):
        trainer_info = fold.get('trainer_info', {})
        history = trainer_info.get('history', [])
        ax = axes[i]
        if not history:
            ax.text(0.5, 0.5, 'no history', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='gray')
            ax.set_title(f"Fold {fold.get('fold_index', i)}")
            continue
        epochs = [h.get('epoch') for h in history]
        train_loss = [h.get('train_loss') for h in history]
        val_loss = [h.get('val_loss') for h in history]
        ax.plot(epochs, train_loss, label='train_loss', color='steelblue', linewidth=1.5)
        ax.plot(epochs, val_loss, label='val_loss', color='darkorange', linewidth=1.5)
        best_epoch = trainer_info.get('best_epoch')
        if best_epoch is not None:
            ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.6,
                       label=f'best ep {best_epoch}')
        ax.set_title(f"Fold {fold.get('fold_index', i)}", fontsize=10)
        ax.set_xlabel('epoch')
        if i == 0:
            ax.set_ylabel('loss')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)
    fig.suptitle(exp_name, fontsize=12, y=1.02)
    plt.tight_layout()
    save_path = RESULTS_DIR / f'loss_{exp_name}.png'
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    print(f'  Kaydedildi: {save_path}')

def plot_loss(model, variant):
    plot_loss_for_experiment(f'tf_{model}_{variant}')

print('=== Loss egrileri (her aday icin 4 fold) ===')
for c in CANDIDATES:
    exp_name = f"tf_{c['model']}_{c['label']}"
    print(f'\n--- {exp_name} ---')
    plot_loss_for_experiment(exp_name)

---
## 6 - Drive'a kaydet (Colab)

`threshold_free_results/` icerigi (artifacts + threshold_free_results.csv + loss_*.png) `Drive/ANN-Project/Threshold_Free_Results/` altina kopyalanir.

In [ ]:
if IN_COLAB:
    drive_dest = Path('/content/drive/MyDrive/ANN-Project/Threshold_Free_Results')
    drive_dest.mkdir(parents=True, exist_ok=True)

    for item in RESULTS_DIR.iterdir():
        dest = drive_dest / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

    print(f'>> Drive: {drive_dest}')
    for p in sorted(drive_dest.rglob('*')):
        if p.is_file():
            print(f'   {p.relative_to(drive_dest)}')
else:
    print(f'Local mode - sonuclar: {RESULTS_DIR}')